In [33]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# ==========================================
# 1. LOAD PREPARED TRAINING AND TESTING DATA
# ==========================================
print("Loading datasets...")
df_train = pd.read_csv("chembl_bcl2_training_set.csv")  # Training data (ChEMBL)
df_test = pd.read_csv("pubchem_bcl2_ml_ready.csv")  # Virtual screen data (PubChem)


Loading datasets...


In [35]:
print(df_train.shape)
print(df_train.columns)
df_train.head()

(2130, 1028)
Index(['molecule_chembl_id', 'canonical_smiles', 'standard_value',
       'Activity_Label', 'Bit_0', 'Bit_1', 'Bit_2', 'Bit_3', 'Bit_4', 'Bit_5',
       ...
       'Bit_1014', 'Bit_1015', 'Bit_1016', 'Bit_1017', 'Bit_1018', 'Bit_1019',
       'Bit_1020', 'Bit_1021', 'Bit_1022', 'Bit_1023'],
      dtype='str', length=1028)


,molecule_chembl_id,canonical_smiles,standard_value,Activity_Label,Bit_0,Bit_1,Bit_2,Bit_3,Bit_4,Bit_5,...,Bit_1014,Bit_1015,Bit_1016,Bit_1017,Bit_1018,Bit_1019,Bit_1020,Bit_1021,Bit_1022,Bit_1023
0,CHEMBL106708,O=C(O)c1ccc(-c2ccc(F)cc2)cc1,300000.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CHEMBL1094250,Cc1cc2c(C(=O)NC[C@H](C)c3ccccc3)c(O)c(O)cc2c(O...,320.0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,CHEMBL1094366,Cc1cc2c(C(=O)NC[C@@H](C)c3ccccc3)c(O)c(O)cc2c(...,430.0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,CHEMBL1095309,COc1ccc(C2C3=C(N=c4s/c(=C\c5ccc(-c6cccc(C(=O)O...,11100.0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,CHEMBL1095310,COc1cccc(C2C3=C(N=c4s/c(=C\c5ccc(-c6cccc(C(=O)...,19400.0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [36]:
print(df_test.shape)
print(df_test.columns)
df_test.head()

(6306, 1031)
Index(['CID', 'ConnectivitySMILES', 'MW', 'LogP', 'HBD', 'HBA',
       'Lipinski_Pass', 'Bit_0', 'Bit_1', 'Bit_2',
       ...
       'Bit_1014', 'Bit_1015', 'Bit_1016', 'Bit_1017', 'Bit_1018', 'Bit_1019',
       'Bit_1020', 'Bit_1021', 'Bit_1022', 'Bit_1023'],
      dtype='str', length=1031)


,CID,ConnectivitySMILES,MW,LogP,HBD,HBA,Lipinski_Pass,Bit_0,Bit_1,Bit_2,...,Bit_1014,Bit_1015,Bit_1016,Bit_1017,Bit_1018,Bit_1019,Bit_1020,Bit_1021,Bit_1022,Bit_1023
0,16196702,CN(C)C(CC1=CC=CC=C1)C(=O)C2=CC=CS2.Cl,295.835,3.5255,0,3,True,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1,1452362,CC(C)N1C(=NN=N1)SCC(=O)N2CCN(CC2)C(=O)C3=CC=CO3,364.431,0.9237,0,7,True,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,4215262,CCC1=CC=C(C=C1)NC(=O)C(C)OC(=O)C2=C(N=CC=C2)OCC,342.395,3.2267,1,5,True,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,2940489,CC1C=CCC2C1C(=O)N(C2=O)C3=CC=C(C=C3)OC(=O)C4=C...,367.426,3.6689,0,5,True,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,3226716,C1CCC(CC1)NC(=O)NCCCN2CCCC2=O,267.373,1.6308,2,2,True,0,0,1,...,0,0,0,0,1,1,0,1,0,0


In [ ]:
# this is
# df_train
# (2130, 1028)
# Index(['molecule_chembl_id', 'canonical_smiles', 'standard_value',
#        'Activity_Label', 'Bit_0', 'Bit_1', 'Bit_2', 'Bit_3', 'Bit_4', 'Bit_5',
#        ...
#        'Bit_1014', 'Bit_1015', 'Bit_1016', 'Bit_1017', 'Bit_1018', 'Bit_1019',
#        'Bit_1020', 'Bit_1021', 'Bit_1022', 'Bit_1023'],
#       dtype='str', length=1028)

# df_test
# (6306, 1031)
# Index(['CID', 'ConnectivitySMILES', 'MW', 'LogP', 'HBD', 'HBA',
#        'Lipinski_Pass', 'Bit_0', 'Bit_1', 'Bit_2',
#        ...
#        'Bit_1014', 'Bit_1015', 'Bit_1016', 'Bit_1017', 'Bit_1018', 'Bit_1019',
#        'Bit_1020', 'Bit_1021', 'Bit_1022', 'Bit_1023'],
#       dtype='str', length=1031)


In [58]:

# Isolate the exact 1024 binary features for X
feature_cols = [f"Bit_{i}" for i in range(1024)]

X_train = df_train[feature_cols].values
y_train = df_train["Activity_Label"].values
X_test = df_test[feature_cols].values

print(f"-> ChEMBL training features shape: {X_train.shape}")
print(f"-> PubChem screening features shape: {X_test.shape}")


# ==========================================
# 2. INITIALIZE AND TRAIN THE MACHINE LEARNING MODEL
# ==========================================
print("\nTraining the Random Forest model on ChEMBL dataset...")
# Using parameters inspired by the paper's optimization goals
model = RandomForestClassifier(
    n_estimators=100, 
    criterion='entropy', 
    max_depth=10, 
    min_samples_split=2, 
    min_samples_leaf=4, 
    random_state=42, 
    n_jobs=-1
)
model.fit(X_train, y_train)
print("-> Model training complete!")



-> ChEMBL training features shape: (2130, 1024)
-> PubChem screening features shape: (6306, 1024)

Training the Random Forest model on ChEMBL dataset...
-> Model training complete!


In [41]:

# ==========================================
# 3. GENERATE ACTIVE PREDICTIONS FOR PUBCHEM DATA
# ==========================================
print("\nScreening PubChem molecules with the trained model...")
# Generate values dynamically back onto the dataframe to prevent KeyErrors
df_test["ML_Predicted_Active"] = model.predict(X_test)
df_test["ML_Active_Probability"] = model.predict_proba(X_test)[:, 1]

total_active = df_test["ML_Predicted_Active"].sum()
print(f"-> The model classified {total_active} out of {len(df_test)} molecules as 'Active'.")



Screening PubChem molecules with the trained model...
-> The model classified 5851 out of 6306 molecules as 'Active'.


In [49]:
# df_test[df_test['ML_Predicted_Active']==1]f

In [50]:

# ==========================================
# 4. CROSS-FILTER WITH LIPINSKI'S RULE OF FIVE
# ==========================================
print("\nCross-filtering active molecules with Lipinski's Filter...")

# Select rows where ML predicted active (1) AND Lipinski_Pass column is True
lead_candidates = df_test[
    (df_test["ML_Predicted_Active"] == 1) & (df_test["Lipinski_Pass"] == True)
].copy()

# Sort by highest active probability score so the strongest hits are on top
lead_candidates = lead_candidates.sort_values(
    by="ML_Active_Probability", ascending=False
)

print(f"\n--- SCREENING PIPELINE RESULTS SUMMARY ---")
print(f"Initial PubChem Pool:                  {len(df_test)} compounds")
print(f"Molecules Predicted Active by ML:       {total_active}")
print(f"Drug-like Lead Candidates:             {len(lead_candidates)}")


# ==========================================
# 5. SAVE THE LEAD CANDIDATES
# ==========================================
output_filename = "bcl2_lead_candidates_docking_ready.csv"
lead_candidates.to_csv(output_filename, index=False)
print(f"\nSuccess! Filtered leads saved to '{output_filename}'")

# Show the top hits
print("\nYour Top 5 Lead Candidates:")
print(
    lead_candidates[
        ["CID", "MW", "LogP", "ML_Active_Probability"]
    ].head()
)


Cross-filtering active molecules with Lipinski's Filter...

--- SCREENING PIPELINE RESULTS SUMMARY ---
Initial PubChem Pool:                  6306 compounds
Molecules Predicted Active by ML:       5851
Drug-like Lead Candidates:             1324

Success! Filtered leads saved to 'bcl2_lead_candidates_docking_ready.csv'

Your Top 5 Lead Candidates:
            CID       MW     LogP  ML_Active_Probability
5415  156294005  492.373  6.00900                    1.0
1067   51003064  408.526  6.29868                    1.0
1006   44570051  551.135  4.98452                    1.0
5425  168291284  491.385  6.00200                    1.0
5423  156294009  491.389  5.53410                    1.0


In [54]:
lead_candidates['ML_Active_Probability'].value_counts()

ML_Active_Probability
0.75    70
0.78    66
0.74    58
0.79    55
0.76    55
0.80    53
0.73    53
0.77    52
0.71    52
0.81    49
0.72    48
0.70    48
0.82    43
0.84    42
0.69    39
0.83    33
0.67    30
0.85    29
0.68    29
0.86    26
0.66    25
1.00    23
0.61    23
0.59    23
0.65    22
0.62    21
0.88    20
0.87    19
0.64    18
0.63    18
0.57    16
0.60    15
0.58    15
0.91    14
0.99    13
0.55    13
0.92    11
0.90    11
0.56    11
0.54    10
0.89     9
0.98     7
0.97     7
0.95     6
0.53     6
0.52     6
0.94     5
0.93     3
0.96     2
0.51     2
Name: count, dtype: int64

In [57]:
lead_candidates.head(10)

,CID,ConnectivitySMILES,MW,LogP,HBD,HBA,Lipinski_Pass,Bit_0,Bit_1,Bit_2,...,Bit_1016,Bit_1017,Bit_1018,Bit_1019,Bit_1020,Bit_1021,Bit_1022,Bit_1023,ML_Predicted_Active,ML_Active_Probability
5415,156294005,CCOC(=O)NC1=CC=C(C=C1)C2=NN(C(C2)C3=CC=CC=C3)C...,492.373,6.00900,1,4,True,0,0,0,...,0,0,0,1,0,0,0,0,1,1.0
1067,51003064,CCC(C)(C)C1=CC=C(C=C1)SC2=C3C=CC=C4C3=C(C=C2)C...,408.526,6.29868,0,3,True,0,1,0,...,0,0,0,0,0,0,0,0,1,1.0
1006,44570051,CCCCN(CCCC)C(=O)C1=NN(C(=C1Cl)C)C2=C(C=C(C=C2)...,551.135,4.98452,2,5,True,0,0,0,...,0,0,0,1,0,0,0,0,1,1.0
5425,168291284,CC(C)C(=O)OC1=CC=C(C=C1)C2=NN(C(C2)C3=CC=CC=C3...,491.385,6.00200,0,4,True,0,1,0,...,0,0,0,1,0,0,0,0,1,1.0
5423,156294009,CN(C)C(=O)NC1=CC=C(C=C1)C2=NN(C(C2)C3=CC=CC=C3...,491.389,5.53410,1,3,True,0,0,0,...,0,0,0,1,0,0,0,0,1,1.0
5419,168288561,CC(C)OC(=O)C1=CC=C(C=C1)C2=NN(C(C2)C3=CC=CC=C3...,491.385,6.00580,0,4,True,0,1,0,...,0,0,0,1,0,0,0,0,1,1.0
1510,145968700,CC1(CCC(=C(C1)CN2CCN(CC2)C3=CC=C(C=C3)C(=O)NS(...,582.170,4.97810,1,6,True,0,0,0,...,0,0,0,0,0,0,0,0,1,1.0
1585,155534540,C1=CC2=C(C=CC3=C2C(=C1)C(=O)C(=C3C#N)C#N)SC4=C...,417.287,5.75056,0,4,True,0,1,0,...,0,0,0,0,0,0,0,0,1,1.0
1247,71717322,CCCOC(=O)C1=C2C3=C4C(=C(C=C3)N5CCSCC5)C=CC=C4C...,392.480,3.44270,0,5,True,0,0,0,...,0,0,0,0,0,0,0,0,1,1.0
1246,71716725,CCOC(=O)C1=C2C3=C4C(=C(C=C3)N5CCSCC5)C=CC=C4C2...,378.453,3.05260,0,5,True,0,0,0,...,0,0,0,0,0,0,0,0,1,1.0


In [64]:
lead_candidates[['CID', 'ConnectivitySMILES']]

,CID,ConnectivitySMILES
5415,156294005,CCOC(=O)NC1=CC=C(C=C1)C2=NN(C(C2)C3=CC=CC=C3)C...
1067,51003064,CCC(C)(C)C1=CC=C(C=C1)SC2=C3C=CC=C4C3=C(C=C2)C...
1006,44570051,CCCCN(CCCC)C(=O)C1=NN(C(=C1Cl)C)C2=C(C=C(C=C2)...
5425,168291284,CC(C)C(=O)OC1=CC=C(C=C1)C2=NN(C(C2)C3=CC=CC=C3...
5423,156294009,CN(C)C(=O)NC1=CC=C(C=C1)C2=NN(C(C2)C3=CC=CC=C3...
...,...,...
4541,117928931,CC1=CC(=CC(=C1Cl)C)OCCCC2=C(N(C3=C2C=CC=C3C4=C...
4368,90085601,CC1=CC(=CC(=C1Cl)C)OCCCC2=C(NC3=CC=CC=C32)C(=O...
1253,71545824,C1=CC=C2C(=C1)C=CC=C2OCCCC3=C(NC4=C3C=CC(=C4)C...
4254,73504862,C1=CC=C(C=C1)CNC2=CC=CC3=C2SC(=C3CCCOC4=CC=CC5...


In [55]:
import pubchempy as pcp

# List out your top CIDs from your terminal printout
top_cids = [156294005, 51003064, 44570051, 168291284, 156294009]

print("--- Fetching Lead Candidate Structural Details From PubChem ---")
for cid in top_cids:
    try:
        # Pull compound metadata from PubChem database
        comp = pcp.Compound.from_cid(cid)
        
        print(f"\n[+] CID: {cid}")
        print(f"    Chemical Name: {comp.iupac_name}")
        print(f"    Molecular Formula: {comp.molecular_formula}")
        print(f"    SMILES Structure: {comp.canonical_smiles}")
        
        # OPTIONAL: Automatically download the 3D SDF structure file for docking!
        # This saves the 3D file you will need to feed into AutoDock Vina / PyMOL later
        pcp.download('SDF', f'candidate_{cid}_3d.sdf', cid, record_type='3d')
        print(f"    -> Saved 3D coordinate file as 'candidate_{cid}_3d.sdf'")
        
    except Exception as e:
        print(f"Could not fetch details for CID {cid}: {e}")

--- Fetching Lead Candidate Structural Details From PubChem ---

[+] CID: 156294005
    Chemical Name: ethyl N-[4-[2-(4-bromobenzoyl)-3-phenyl-3,4-dihydropyrazol-5-yl]phenyl]carbamate
    Molecular Formula: C25H22BrN3O3
    SMILES Structure: CCOC(=O)NC1=CC=C(C=C1)C2=NN(C(C2)C3=CC=CC=C3)C(=O)C4=CC=C(C=C4)Br


C:\Users\Adnan Habib\AppData\Local\Temp\ipykernel_27420\2647260038.py:15: PubChemPyDeprecationWarning: canonical_smiles is deprecated: Use connectivity_smiles instead
  print(f"    SMILES Structure: {comp.canonical_smiles}")


    -> Saved 3D coordinate file as 'candidate_156294005_3d.sdf'

[+] CID: 51003064
    Chemical Name: 3-[4-(2-methylbutan-2-yl)phenyl]sulfanyl-8-oxoacenaphthyleno[1,2-b]pyrrole-9-carbonitrile
    Molecular Formula: C26H20N2OS
    SMILES Structure: CCC(C)(C)C1=CC=C(C=C1)SC2=C3C=CC=C4C3=C(C=C2)C5=C(C(=O)N=C45)C#N
    -> Saved 3D coordinate file as 'candidate_51003064_3d.sdf'

[+] CID: 44570051
    Chemical Name: 1-[4-amino-2-[(3S)-3-(aminomethyl)-3,4-dihydro-1H-isoquinoline-2-carbonyl]phenyl]-N,N-dibutyl-4-chloro-5-methylpyrazole-3-carboxamide
    Molecular Formula: C30H39ClN6O2
    SMILES Structure: CCCCN(CCCC)C(=O)C1=NN(C(=C1Cl)C)C2=C(C=C(C=C2)N)C(=O)N3CC4=CC=CC=C4CC3CN
    -> Saved 3D coordinate file as 'candidate_44570051_3d.sdf'

[+] CID: 168291284
    Chemical Name: [4-[2-(4-bromobenzoyl)-3-phenyl-3,4-dihydropyrazol-5-yl]phenyl] 2-methylpropanoate
    Molecular Formula: C26H23BrN2O3
    SMILES Structure: CC(C)C(=O)OC1=CC=C(C=C1)C2=NN(C(C2)C3=CC=CC=C3)C(=O)C4=CC=C(C=C4)Br
    -> Sav

In [ ]:

# The Training Phase:
#     You showed the model 2,130 molecules from the ChEMBL database.
# For each molecule, you gave it the 1,024-bit fingerprint (the chemical features) and
# told it the answer (1 for Active against Bcl-2, 0 for Inactive). The model studied these patterns
# and learned exactly which chemical groupings make a molecule a strong weapon against the Bcl-2 
# cancer protein.

# The Testing (Screening) Phase:
#     You gave the model 6,306 brand-new, completely
# unknown molecules from PubChem. The model scanned their 1,024-bit fingerprints and
# asked itself: "Based on my training, does this look like a winner?" ---

# They all share a dihydropyrazol core group bound to a bromobenzoyl group.

In [66]:
import os
from rdkit import Chem
from rdkit.Chem import AllChem

# 1. Define your discovered lead CIDs
top_cids = [156294005, 51003064, 44570051, 168291284, 156294009]

print("=== Starting Step 1: 3D Geometry Optimization (MMFF94) ===\n")

for cid in top_cids:
    input_filename = f'candidate_{cid}_3d.sdf'
    output_filename = f'candidate_{cid}_optimized.sdf'
    
    # Verify the downloaded file exists
    if not os.path.exists(input_filename):
        print(f"[!] Warning: {input_filename} not found. Skipping...")
        continue
        
    # Read the molecule from the downloaded file
    # (SDMolSupplier handles raw spatial coordination streams)
    suppl = Chem.SDMolSupplier(input_filename)
    mol = suppl[0]
    
    if mol is None:
        print(f"[X] Error: Could not read molecular coordinates for CID {cid}")
        continue
        
    print(f"[+] Processing CID {cid} ({mol.GetNumAtoms()} atoms)...")
    
    # A. Add implicit Hydrogens (Crucial! Force fields require H-atoms for exact electrostatic math)
    mol_with_h = Chem.AddHs(mol)
    
    # B. Initialize the Merck Molecular Force Field (MMFF94)
    # The paper explicitly uses MMFF94 for small-molecule ligand optimization
    mp = AllChem.MMFFGetMoleculeProperties(mol_with_h, mmffVariant='MMFF94')
    ff = AllChem.MMFFGetMoleculeForceField(mol_with_h, mp)
    
    if ff is None:
        print(f"    [-] MMFF94 parameters not applicable. Falling back to universal UFF...")
        ff = AllChem.UFFGetMoleculeForceField(mol_with_h)
        
if ff:
        # Calculate initial crude physical energy
        initial_energy = ff.CalcEnergy()
        print(f"    -> Initial Potential Energy: {initial_energy:.2f} kcal/mol")
        
        # C. Execute the structural minimization loop safely
        try:
            # We add a pre-cleaning geometry cleanup step to prevent linearSearch failure
            AllChem.EmbedMolecule(mol_with_h, randomSeed=42, useExpTorsionAnglePrefs=True, useBasicKnowledge=True)
            
            # Re-initialize the forcefield on the cleaned layout
            mp = AllChem.MMFFGetMoleculeProperties(mol_with_h, mmffVariant='MMFF94')
            ff = AllChem.MMFFGetMoleculeForceField(mol_with_h, mp) or AllChem.UFFGetMoleculeForceField(mol_with_h)
            
            convergence_status = ff.Minimize(maxIts=500)
            
            # Calculate optimized physical energy
            final_energy = ff.CalcEnergy()
            print(f"    -> Optimized Potential Energy: {final_energy:.2f} kcal/mol")
            print(f"    -> Energy Dropped by: {initial_energy - final_energy:.2f} kcal/mol")
            
            # D. Save the relaxed, optimized molecule
            writer = Chem.SDWriter(output_filename)
            writer.write(mol_with_h)
            writer.close()
            print(f"    [✔] Successfully saved clean file: '{output_filename}'\n")
            
        except RuntimeError as e:
            # Catch the linear search error and force fall back to the ultra-flexible UFF forcefield
            print(f"    [!] MMFF94 math failure. Forcing Universal Force Field (UFF) rescue...")
            try:
                ff_uff = AllChem.UFFGetMoleculeForceField(mol_with_h)
                ff_uff.Minimize(maxIts=500)
                final_energy = ff_uff.CalcEnergy()
                print(f"    -> UFF Optimized Potential Energy: {final_energy:.2f} kcal/mol")
                
                writer = Chem.SDWriter(output_filename)
                writer.write(mol_with_h)
                writer.close()
                print(f"    [✔] Saved via UFF rescue filter: '{output_filename}'\n")
            except Exception as inner_e:
                print(f"    [X] Molecule coordinates completely un-optimizable: {inner_e}")

=== Starting Step 1: 3D Geometry Optimization (MMFF94) ===

[+] Processing CID 156294005 (32 atoms)...
[+] Processing CID 51003064 (30 atoms)...
[+] Processing CID 44570051 (39 atoms)...
[+] Processing CID 168291284 (32 atoms)...
[+] Processing CID 156294009 (32 atoms)...
    -> Initial Potential Energy: 7588830742.00 kcal/mol
    -> Optimized Potential Energy: 54.07 kcal/mol
    -> Energy Dropped by: 7588830687.93 kcal/mol
    [✔] Successfully saved clean file: 'candidate_156294009_optimized.sdf'



In [69]:
import os
from rdkit import Chem
from meeko import MoleculePreparation
from meeko import PDBQTWriterLegacy

# 1. Define your target optimized lead CIDs
top_cids = [156294005, 51003064, 44570051, 168291284, 156294009]

print("=== Starting Step 2: Ligand Conversion to PDBQT format ===\n")

for cid in top_cids:
    input_filename = f'candidate_{cid}_optimized.sdf'
    output_filename = f'candidate_{cid}.pdbqt'
    
    if not os.path.exists(input_filename):
        print(f"[!] Warning: Optimized file {input_filename} not found. Skipping...")
        continue
        
    # Read optimized molecule (ensure removeHs=False to preserve our precious implicit hydrogens!)
    for mol in Chem.SDMolSupplier(input_filename, removeHs=False):
        if mol is None:
            continue
            
        print(f"[+] Formatting CID {cid} to PDBQT layout...")
        
        # Initialize Meeko's parameterizer
        preparator = MoleculePreparation()
        
        # Prepare the structural data tree (assigns Gasteiger charges and flags rotatable bonds)
        mol_setups = preparator.prepare(mol)
        
        if mol_setups:
            setup = mol_setups[0]
            
            # FIX: Safely extract just the string component from Meeko's return tuple
            pdbqt_output = PDBQTWriterLegacy.write_string(setup)
            pdbqt_string = pdbqt_output[0] if isinstance(pdbqt_output, tuple) else pdbqt_output
            
            # Write out to disk
            with open(output_filename, 'w') as f:
                f.write(pdbqt_string)
                
            print(f"    [✔] Successfully generated: '{output_filename}'")
        else:
            print(f"    [X] Meeko could not parameterize structure layout for CID {cid}")
            
print("\n=== Conversion Pipeline Complete! Your 5 Ligands are Docking-Ready ===")

=== Starting Step 2: Ligand Conversion to PDBQT format ===

[+] Formatting CID 156294005 to PDBQT layout...
    [✔] Successfully generated: 'candidate_156294005.pdbqt'
[+] Formatting CID 51003064 to PDBQT layout...
    [✔] Successfully generated: 'candidate_51003064.pdbqt'
[!] Warning: Optimized file candidate_44570051_optimized.sdf not found. Skipping...
[!] Warning: Optimized file candidate_168291284_optimized.sdf not found. Skipping...
[+] Formatting CID 156294009 to PDBQT layout...
    [✔] Successfully generated: 'candidate_156294009.pdbqt'

=== Conversion Pipeline Complete! Your 5 Ligands are Docking-Ready ===
